<a href="https://colab.research.google.com/github/NikKimo/teaching/blob/main/TFP_and_DEA_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Load packages

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display, HTML
from scipy.optimize import linprog
import matplotlib.pyplot as plt

#Generate synthetic data

##Airport operational data

In [ ]:
# ============================================================
# Generate synthetic airport operational data
# with variable returns to scale
# ============================================================

rng = np.random.default_rng(42)

n = 20


# ------------------------------------------------------------
# Underlying airport size
#
# Not sorted:
# A1 to A20 are just airport IDs, not ordered by size
# ------------------------------------------------------------

airport_size = rng.uniform(
    0.4,
    2.4,
    n
)


# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

fte = np.round(
    1000
    * airport_size
    * rng.uniform(0.85, 1.15, n)
).astype(int)


gates = np.round(
    35
    * airport_size
    * rng.uniform(0.80, 1.20, n)
).astype(int)


terminal_area = np.round(
    90_000
    * airport_size
    * rng.uniform(0.80, 1.20, n)
).astype(int)


# ------------------------------------------------------------
# Airport-specific operational efficiency
# ------------------------------------------------------------

efficiency = rng.uniform(
    0.70,
    1.00,
    n
)


# ------------------------------------------------------------
# Scale effect
# ------------------------------------------------------------

scale_effect = np.exp(
    -0.40
    * np.log(
        airport_size / 1.10
    )**2
)


# ------------------------------------------------------------
# Outputs
# ------------------------------------------------------------

# Passengers, million
passengers = (
    0.004
    * fte**0.50
    * gates**0.30
    * terminal_area**0.20
    * efficiency
    * scale_effect
)


# Aircraft movements, thousand
movements = (
    0.20
    * fte**0.35
    * gates**0.45
    * terminal_area**0.20
    * efficiency
    * scale_effect
)


# Cargo, thousand tonnes
cargo = (
    0.03
    * fte**0.30
    * gates**0.20
    * terminal_area**0.50
    * efficiency
    * scale_effect
)


# ------------------------------------------------------------
# Deliberate output shocks for teaching
#
# Same inputs, but lower outputs
# ------------------------------------------------------------

# A1: 10% lower outputs
passengers[0] *= 0.90
movements[0] *= 0.90
cargo[0] *= 0.90


# A10: 20% lower outputs
passengers[9] *= 0.80
movements[9] *= 0.80
cargo[9] *= 0.80


# A15: 30% lower outputs
passengers[14] *= 0.70
movements[14] *= 0.70
cargo[14] *= 0.70


# ------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------

airport_operational_data = pd.DataFrame({

    "Airport": [
        f"A{i+1}"
        for i in range(n)
    ],

    "FTE": fte,

    "Gates": gates,

    "Terminal area (m²)": terminal_area,

    "Passengers (million)": passengers,

    "Aircraft movements (000)": movements,

    "Cargo (000 tonnes)": cargo
})

,Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes)
0,A1,2099,66,187121,5.185255,154.301393,211.403055
1,A2,1222,51,113669,3.801233,118.023733,152.583339
2,A3,2416,80,195520,6.658260,199.677769,262.331830
3,A4,2006,58,178648,4.635913,136.229079,190.540360
4,A5,637,23,55805,1.390815,42.254945,55.632914
5,A6,2136,92,216147,4.833334,150.780379,198.350744
6,A7,1903,64,177102,5.512904,165.714933,226.153343
7,A8,1702,63,163573,4.617559,140.811980,189.435442
8,A9,588,25,47976,1.479466,46.063760,56.990700
9,A10,1372,39,114106,3.071176,90.018395,123.874299


##Airport economic data

In [ ]:
airport_economic_data = pd.DataFrame({

    "Airport": airport_operational_data["Airport"],

    # Input prices / costs
    "Salary per FTE ($)": rng.integers(
        65_000, 100_000, n
    ),

    "Annual cost per gate ($)": rng.integers(
        350_000, 650_000, n
    ),

    "Annual cost per m² terminal ($)": rng.integers(
        350, 650, n
    ),

    # Output prices / revenues
    "Revenue per passenger ($)": rng.uniform(
        15, 30, n
    ),

    "Revenue per movement ($)": rng.integers(
        700, 1500, n
    ),

    "Revenue per tonne cargo ($)": rng.uniform(
        30, 80, n
    )
})

# Load data

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#airport_operational_data = pd.read_excel('/content/drive/MyDrive/teaching/raw_data/airport_operational_data.xlsx')
#airport_economic_data = pd.read_excel('/content/drive/MyDrive/teaching/raw_data/airport_economic_data.xlsx')

#Inspect data

In [ ]:
airport_operational_data

,Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes)
0,A1,2099,66,187121,5.185255,154.301393,211.403055
1,A2,1222,51,113669,3.801233,118.023733,152.583339
2,A3,2416,80,195520,6.658260,199.677769,262.331830
3,A4,2006,58,178648,4.635913,136.229079,190.540360
4,A5,637,23,55805,1.390815,42.254945,55.632914
5,A6,2136,92,216147,4.833334,150.780379,198.350744
6,A7,1903,64,177102,5.512904,165.714933,226.153343
7,A8,1702,63,163573,4.617559,140.811980,189.435442
8,A9,588,25,47976,1.479466,46.063760,56.990700
9,A10,1372,39,114106,3.071176,90.018395,123.874299


In [ ]:
airport_economic_data

,Airport,Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($)
0,A1,77819,388273,392,17.558895,990,35.387047
1,A2,96800,486732,525,28.876802,1361,75.800592
2,A3,82327,561209,378,23.715917,707,41.510700
3,A4,89489,410709,544,20.203047,1416,31.870628
4,A5,81009,566484,538,23.863732,1134,57.742623
5,A6,74305,441786,375,15.342058,812,48.546114
6,A7,91748,592598,577,29.378388,941,71.489487
7,A8,98921,523765,474,22.234552,1143,70.412574
8,A9,74223,514072,586,26.741028,952,45.856945
9,A10,92256,403031,362,16.240950,786,77.644970


Resources consumed: Labor + Aircraft capacity + Terminal capacity​

Services produced: Passengers + Aircraft movements + Cargo

Note: Doubling FTE does not double passenger output.

# Calculate TFP

# TFP Calculation

## 1. Economic value of inputs

$$
C_{L,i} = FTE_i \times Salary_i
$$

$$
C_{G,i} = Gates_i \times CostPerGate_i
$$

$$
C_{T,i} = TerminalArea_i \times CostPerArea_i
$$

## 2. Economic value of outputs

$$
R_{P,i} = Passengers_i \times RevenuePerPassenger_i
$$

$$
R_{M,i} = Movements_i \times RevenuePerMovement_i
$$

$$
R_{C,i} = Cargo_i \times RevenuePerCargo_i
$$

## 3. Normalize physical quantities

For any variable $X$:

$$
X_{n,i} = \frac{X_i}{\bar{X}}
$$

For example:

$$
FTE_{n,i} = \frac{FTE_i}{\overline{FTE}}
$$

$$
Passengers_{n,i} =
\frac{Passengers_i}{\overline{Passengers}}
$$

A normalized value of $1$ represents the sample mean.

## 4. Total input cost

$$
C_{\mathrm{total}}
=
\sum_i C_{L,i}
+
\sum_i C_{G,i}
+
\sum_i C_{T,i}
$$

## 5. Common input weights

$$
w_L =
\frac{\sum_i C_{L,i}}
{C_{\mathrm{total}}}
$$

$$
w_G =
\frac{\sum_i C_{G,i}}
{C_{\mathrm{total}}}
$$

$$
w_T =
\frac{\sum_i C_{T,i}}
{C_{\mathrm{total}}}
$$

$$
w_L + w_G + w_T = 1
$$

## 6. Total output revenue

$$
R_{\mathrm{total}}
=
\sum_i R_{P,i}
+
\sum_i R_{M,i}
+
\sum_i R_{C,i}
$$

## 7. Common output weights

$$
w_P =
\frac{\sum_i R_{P,i}}
{R_{\mathrm{total}}}
$$

$$
w_M =
\frac{\sum_i R_{M,i}}
{R_{\mathrm{total}}}
$$

$$
w_C =
\frac{\sum_i R_{C,i}}
{R_{\mathrm{total}}}
$$

$$
w_P + w_M + w_C = 1
$$

## 8. Input index

$$
I_i
=
FTE_{n,i}^{w_L}
\times
Gates_{n,i}^{w_G}
\times
Terminal_{n,i}^{w_T}
$$

## 9. Output index

$$
O_i
=
Passengers_{n,i}^{w_P}
\times
Movements_{n,i}^{w_M}
\times
Cargo_{n,i}^{w_C}
$$

## 10. Total Factor Productivity

$$
\boxed{
TFP_i = \frac{O_i}{I_i}
}
$$

or equivalently

$$
\boxed{
TFP_i
=
\frac{
Passengers_{n,i}^{w_P}
Movements_{n,i}^{w_M}
Cargo_{n,i}^{w_C}
}{
FTE_{n,i}^{w_L}
Gates_{n,i}^{w_G}
Terminal_{n,i}^{w_T}
}
}
$$

In [ ]:
# Merge operational and economic data
tfp_data = airport_operational_data.merge(
    airport_economic_data,
    on="Airport"
)

In [ ]:
tfp_data

,Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($)
0,A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047
1,A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592
2,A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700
3,A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628
4,A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623
5,A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114
6,A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487
7,A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574
8,A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945
9,A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970


In [ ]:
# ------------------------------------------------------------
# Calculate total economic value of each input and output
# ------------------------------------------------------------

tfp_data["Labor cost"] = (
    tfp_data["FTE"]
    * tfp_data["Salary per FTE ($)"]
)

tfp_data["Gate cost"] = (
    tfp_data["Gates"]
    * tfp_data["Annual cost per gate ($)"]
)

tfp_data["Terminal cost"] = (
    tfp_data["Terminal area (m²)"]
    * tfp_data["Annual cost per m² terminal ($)"]
)

tfp_data["Passenger revenue"] = (
    tfp_data["Passengers (million)"]
    * 1_000_000
    * tfp_data["Revenue per passenger ($)"]
)

tfp_data["Movement revenue"] = (
    tfp_data["Aircraft movements (000)"]
    * 1_000
    * tfp_data["Revenue per movement ($)"]
)

tfp_data["Cargo revenue"] = (
    tfp_data["Cargo (000 tonnes)"]
    * 1_000
    * tfp_data["Revenue per tonne cargo ($)"]
)

In [ ]:
tfp_data

,Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($),Labor cost,Gate cost,Terminal cost,Passenger revenue,Movement revenue,Cargo revenue
0,A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047,163342081,25626018,73351432,9.104735e+07,1.527584e+08,7.480930e+06
1,A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592,118289600,24823332,59676225,1.097675e+08,1.606303e+08,1.156591e+07
2,A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700,198902032,44896720,73906560,1.579067e+08,1.411722e+08,1.088958e+07
3,A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628,179514934,23821122,97184512,9.365956e+07,1.929004e+08,6.072641e+06
4,A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623,51602733,13029132,30023090,3.319003e+07,4.791711e+07,3.212390e+06
5,A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114,158715480,40644312,81055125,7.415330e+07,1.224337e+08,9.629158e+06
6,A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487,174596444,37926272,102187854,1.619602e+08,1.559378e+08,1.616759e+07
7,A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574,168363542,32997195,77533602,1.026694e+08,1.609481e+08,1.333864e+07
8,A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945,43643124,12851800,28113936,3.956244e+07,4.385270e+07,2.613419e+06
9,A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970,126575232,15718209,41306372,4.987881e+07,7.075446e+07,9.618216e+06


In [ ]:
# ------------------------------------------------------------
# Normalize physical quantities relative to sample mean (% of mean)
# ------------------------------------------------------------

tfp_data["FTE_n"] = (
    tfp_data["FTE"] / tfp_data["FTE"].mean()
)

tfp_data["Gates_n"] = (
    tfp_data["Gates"] / tfp_data["Gates"].mean()
)

tfp_data["Terminal_n"] = (
    tfp_data["Terminal area (m²)"]
    / tfp_data["Terminal area (m²)"].mean()
)

tfp_data["Passengers_n"] = (
    tfp_data["Passengers (million)"]
    / tfp_data["Passengers (million)"].mean()
)

tfp_data["Movements_n"] = (
    tfp_data["Aircraft movements (000)"]
    / tfp_data["Aircraft movements (000)"].mean()
)

tfp_data["Cargo_n"] = (
    tfp_data["Cargo (000 tonnes)"]
    / tfp_data["Cargo (000 tonnes)"].mean()
)

In [ ]:
display(
    HTML(
        f"""
        <div style="
            max-width: 100%;
            max-height: 600px;
            overflow: auto;
            border: 1px solid #555;
        ">
            {tfp_data.to_html(
                index=False,
                max_cols=None,
                max_rows=None
            )}
        </div>
        """
    )
)

Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($),Labor cost,Gate cost,Terminal cost,Passenger revenue,Movement revenue,Cargo revenue,FTE_n,Gates_n,Terminal_n,Passengers_n,Movements_n,Cargo_n
A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047,163342081,25626018,73351432,9.104735e+07,1.527584e+08,7.480930e+06,1.354499,1.239437,1.350787,1.298821,1.282881,1.307788
A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592,118289600,24823332,59676225,1.097675e+08,1.606303e+08,1.156591e+07,0.788565,0.957746,0.820552,0.952146,0.981264,0.943916
A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700,198902032,44896720,73906560,1.579067e+08,1.411722e+08,1.088958e+07,1.559062,1.502347,1.411418,1.667785,1.660146,1.622846
A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628,179514934,23821122,97184512,9.365956e+07,1.929004e+08,6.072641e+06,1.294486,1.089202,1.289622,1.161220,1.132625,1.178727
A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623,51602733,13029132,30023090,3.319003e+07,4.791711e+07,3.212390e+06,0.411061,0.431925,0.402844,0.348376,0.351313,0.344158
A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114,158715480,40644312,81055125,7.415330e+07,1.224337e+08,9.629158e+06,1.378376,1.727700,1.560319,1.210671,1.253607,1.227044
A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487,174596444,37926272,102187854,1.619602e+08,1.559378e+08,1.616759e+07,1.228019,1.201878,1.278462,1.380892,1.377774,1.399037
A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574,168363542,32997195,77533602,1.026694e+08,1.609481e+08,1.333864e+07,1.098313,1.183099,1.180799,1.156623,1.170728,1.171892
A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945,43643124,12851800,28113936,3.956244e+07,4.385270e+07,2.613419e+06,0.379441,0.469484,0.346329,0.370582,0.382980,0.352558
A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970,126575232,15718209,41306372,4.987881e+07,7.075446e+07,9.618216e+06,0.885361,0.732394,0.823707,0.769279,0.748424,0.766315


In [ ]:
# ------------------------------------------------------------
# Common cross-sectional INPUT weights
# Same value for every airport
# ------------------------------------------------------------

total_input_cost_all = (
    tfp_data["Labor cost"].sum()
    + tfp_data["Gate cost"].sum()
    + tfp_data["Terminal cost"].sum()
)

tfp_data["w_FTE_common"] = (
    tfp_data["Labor cost"].sum()
    / total_input_cost_all
)

tfp_data["w_Gates_common"] = (
    tfp_data["Gate cost"].sum()
    / total_input_cost_all
)

tfp_data["w_Terminal_common"] = (
    tfp_data["Terminal cost"].sum()
    / total_input_cost_all
)


# ------------------------------------------------------------
# Common cross-sectional OUTPUT weights
# Same value for every airport
# ------------------------------------------------------------

total_output_revenue_all = (
    tfp_data["Passenger revenue"].sum()
    + tfp_data["Movement revenue"].sum()
    + tfp_data["Cargo revenue"].sum()
)

tfp_data["w_Passengers_common"] = (
    tfp_data["Passenger revenue"].sum()
    / total_output_revenue_all
)

tfp_data["w_Movements_common"] = (
    tfp_data["Movement revenue"].sum()
    / total_output_revenue_all
)

tfp_data["w_Cargo_common"] = (
    tfp_data["Cargo revenue"].sum()
    / total_output_revenue_all
)

In [ ]:
display(
    HTML(
        f"""
        <div style="
            max-width: 100%;
            max-height: 600px;
            overflow: auto;
            border: 1px solid #555;
        ">
            {tfp_data.to_html(
                index=False,
                max_cols=None,
                max_rows=None
            )}
        </div>
        """
    )
)

Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($),Labor cost,Gate cost,Terminal cost,Passenger revenue,Movement revenue,Cargo revenue,FTE_n,Gates_n,Terminal_n,Passengers_n,Movements_n,Cargo_n,w_FTE_common,w_Gates_common,w_Terminal_common,w_Passengers_common,w_Movements_common,w_Cargo_common
A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047,163342081,25626018,73351432,9.104735e+07,1.527584e+08,7.480930e+06,1.354499,1.239437,1.350787,1.298821,1.282881,1.307788,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592,118289600,24823332,59676225,1.097675e+08,1.606303e+08,1.156591e+07,0.788565,0.957746,0.820552,0.952146,0.981264,0.943916,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700,198902032,44896720,73906560,1.579067e+08,1.411722e+08,1.088958e+07,1.559062,1.502347,1.411418,1.667785,1.660146,1.622846,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628,179514934,23821122,97184512,9.365956e+07,1.929004e+08,6.072641e+06,1.294486,1.089202,1.289622,1.161220,1.132625,1.178727,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623,51602733,13029132,30023090,3.319003e+07,4.791711e+07,3.212390e+06,0.411061,0.431925,0.402844,0.348376,0.351313,0.344158,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114,158715480,40644312,81055125,7.415330e+07,1.224337e+08,9.629158e+06,1.378376,1.727700,1.560319,1.210671,1.253607,1.227044,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487,174596444,37926272,102187854,1.619602e+08,1.559378e+08,1.616759e+07,1.228019,1.201878,1.278462,1.380892,1.377774,1.399037,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574,168363542,32997195,77533602,1.026694e+08,1.609481e+08,1.333864e+07,1.098313,1.183099,1.180799,1.156623,1.170728,1.171892,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945,43643124,12851800,28113936,3.956244e+07,4.385270e+07,2.613419e+06,0.379441,0.469484,0.346329,0.370582,0.382980,0.352558,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879
A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970,126575232,15718209,41306372,4.987881e+07,7.075446e+07,9.618216e+06,0.885361,0.732394,0.823707,0.769279,0.748424,0.766315,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879


In [ ]:
tfp_data["Input index"] = np.exp(
    tfp_data["w_FTE_common"] * np.log(tfp_data["FTE_n"])
    + tfp_data["w_Gates_common"] * np.log(tfp_data["Gates_n"])
    + tfp_data["w_Terminal_common"] * np.log(tfp_data["Terminal_n"])
)

tfp_data["Output index"] = np.exp(
    tfp_data["w_Passengers_common"] * np.log(tfp_data["Passengers_n"])
    + tfp_data["w_Movements_common"] * np.log(tfp_data["Movements_n"])
    + tfp_data["w_Cargo_common"] * np.log(tfp_data["Cargo_n"])
)

In [ ]:
display(
    HTML(
        f"""
        <div style="
            max-width: 100%;
            max-height: 600px;
            overflow: auto;
            border: 1px solid #555;
        ">
            {tfp_data.to_html(
                index=False,
                max_cols=None,
                max_rows=None
            )}
        </div>
        """
    )
)

Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($),Labor cost,Gate cost,Terminal cost,Passenger revenue,Movement revenue,Cargo revenue,FTE_n,Gates_n,Terminal_n,Passengers_n,Movements_n,Cargo_n,w_FTE_common,w_Gates_common,w_Terminal_common,w_Passengers_common,w_Movements_common,w_Cargo_common,Input index,Output index
A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047,163342081,25626018,73351432,9.104735e+07,1.527584e+08,7.480930e+06,1.354499,1.239437,1.350787,1.298821,1.282881,1.307788,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.339170,1.290030
A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592,118289600,24823332,59676225,1.097675e+08,1.606303e+08,1.156591e+07,0.788565,0.957746,0.820552,0.952146,0.981264,0.943916,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.816337,0.968364
A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700,198902032,44896720,73906560,1.579067e+08,1.411722e+08,1.088958e+07,1.559062,1.502347,1.411418,1.667785,1.660146,1.622846,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.508473,1.661599
A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628,179514934,23821122,97184512,9.365956e+07,1.929004e+08,6.072641e+06,1.294486,1.089202,1.289622,1.161220,1.132625,1.178727,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.266715,1.145455
A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623,51602733,13029132,30023090,3.319003e+07,4.791711e+07,3.212390e+06,0.411061,0.431925,0.402844,0.348376,0.351313,0.344158,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.411107,0.349883
A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114,158715480,40644312,81055125,7.415330e+07,1.224337e+08,9.629158e+06,1.378376,1.727700,1.560319,1.210671,1.253607,1.227044,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.467353,1.235716
A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487,174596444,37926272,102187854,1.619602e+08,1.559378e+08,1.616759e+07,1.228019,1.201878,1.278462,1.380892,1.377774,1.399037,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.239109,1.379825
A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574,168363542,32997195,77533602,1.026694e+08,1.609481e+08,1.333864e+07,1.098313,1.183099,1.180799,1.156623,1.170728,1.171892,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.131383,1.165282
A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945,43643124,12851800,28113936,3.956244e+07,4.385270e+07,2.613419e+06,0.379441,0.469484,0.346329,0.370582,0.382980,0.352558,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.379135,0.376876
A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970,126575232,15718209,41306372,4.987881e+07,7.075446e+07,9.618216e+06,0.885361,0.732394,0.823707,0.769279,0.748424,0.766315,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.847793,0.757160


In [ ]:
tfp_data["TFP"] = (
    tfp_data["Output index"]
    / tfp_data["Input index"]
)

In [ ]:
display(
    HTML(
        f"""
        <div style="
            max-width: 100%;
            max-height: 600px;
            overflow: auto;
            border: 1px solid #555;
        ">
            {tfp_data.to_html(
                index=False,
                max_cols=None,
                max_rows=None
            )}
        </div>
        """
    )
)

Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes),Salary per FTE ($),Annual cost per gate ($),Annual cost per m² terminal ($),Revenue per passenger ($),Revenue per movement ($),Revenue per tonne cargo ($),Labor cost,Gate cost,Terminal cost,Passenger revenue,Movement revenue,Cargo revenue,FTE_n,Gates_n,Terminal_n,Passengers_n,Movements_n,Cargo_n,w_FTE_common,w_Gates_common,w_Terminal_common,w_Passengers_common,w_Movements_common,w_Cargo_common,Input index,Output index,TFP
A1,2099,66,187121,5.185255,154.301393,211.403055,77819,388273,392,17.558895,990,35.387047,163342081,25626018,73351432,9.104735e+07,1.527584e+08,7.480930e+06,1.354499,1.239437,1.350787,1.298821,1.282881,1.307788,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.339170,1.290030,0.963306
A2,1222,51,113669,3.801233,118.023733,152.583339,96800,486732,525,28.876802,1361,75.800592,118289600,24823332,59676225,1.097675e+08,1.606303e+08,1.156591e+07,0.788565,0.957746,0.820552,0.952146,0.981264,0.943916,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.816337,0.968364,1.186231
A3,2416,80,195520,6.658260,199.677769,262.331830,82327,561209,378,23.715917,707,41.510700,198902032,44896720,73906560,1.579067e+08,1.411722e+08,1.088958e+07,1.559062,1.502347,1.411418,1.667785,1.660146,1.622846,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.508473,1.661599,1.101511
A4,2006,58,178648,4.635913,136.229079,190.540360,89489,410709,544,20.203047,1416,31.870628,179514934,23821122,97184512,9.365956e+07,1.929004e+08,6.072641e+06,1.294486,1.089202,1.289622,1.161220,1.132625,1.178727,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.266715,1.145455,0.904272
A5,637,23,55805,1.390815,42.254945,55.632914,81009,566484,538,23.863732,1134,57.742623,51602733,13029132,30023090,3.319003e+07,4.791711e+07,3.212390e+06,0.411061,0.431925,0.402844,0.348376,0.351313,0.344158,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.411107,0.349883,0.851077
A6,2136,92,216147,4.833334,150.780379,198.350744,74305,441786,375,15.342058,812,48.546114,158715480,40644312,81055125,7.415330e+07,1.224337e+08,9.629158e+06,1.378376,1.727700,1.560319,1.210671,1.253607,1.227044,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.467353,1.235716,0.842140
A7,1903,64,177102,5.512904,165.714933,226.153343,91748,592598,577,29.378388,941,71.489487,174596444,37926272,102187854,1.619602e+08,1.559378e+08,1.616759e+07,1.228019,1.201878,1.278462,1.380892,1.377774,1.399037,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.239109,1.379825,1.113563
A8,1702,63,163573,4.617559,140.811980,189.435442,98921,523765,474,22.234552,1143,70.412574,168363542,32997195,77533602,1.026694e+08,1.609481e+08,1.333864e+07,1.098313,1.183099,1.180799,1.156623,1.170728,1.171892,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,1.131383,1.165282,1.029963
A9,588,25,47976,1.479466,46.063760,56.990700,74223,514072,586,26.741028,952,45.856945,43643124,12851800,28113936,3.956244e+07,4.385270e+07,2.613419e+06,0.379441,0.469484,0.346329,0.370582,0.382980,0.352558,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.379135,0.376876,0.994040
A10,1372,39,114106,3.071176,90.018395,123.874299,92256,403031,362,16.240950,786,77.644970,126575232,15718209,41306372,4.987881e+07,7.075446e+07,9.618216e+06,0.885361,0.732394,0.823707,0.769279,0.748424,0.766315,0.593552,0.119338,0.28711,0.38794,0.572181,0.039879,0.847793,0.757160,0.893095


##Summary

In [ ]:
tfp_ranking = (
    tfp_data[
        ["Airport", "Input index", "Output index", "TFP"]
    ]
    .sort_values("TFP", ascending=False)
    .reset_index(drop=True)
)

tfp_ranking["Rank"] = tfp_ranking.index + 1

display(tfp_ranking)

,Airport,Input index,Output index,TFP,Rank
0,A16,0.514978,0.654717,1.271350,1
1,A20,1.121552,1.355743,1.208810,2
2,A2,0.816337,0.968364,1.186231,3
3,A7,1.239109,1.379825,1.113563,4
4,A13,1.115171,1.240285,1.112193,5
5,A17,0.888141,0.985471,1.109588,6
6,A3,1.508473,1.661599,1.101511,7
7,A8,1.131383,1.165282,1.029963,8
8,A14,1.262442,1.267972,1.004381,9
9,A11,0.733286,0.732773,0.999301,10


#Calculate DEA

# DEA Calculation

## 1. Define inputs and outputs

For airport \(i\), the input vector is:

$$
x_i =
\begin{bmatrix}
FTE_i \\
Gates_i \\
TerminalArea_i
\end{bmatrix}
$$

The output vector is:

$$
y_i =
\begin{bmatrix}
Passengers_i \\
Movements_i \\
Cargo_i
\end{bmatrix}
$$

---

## 2. Construct a synthetic benchmark

DEA constructs a benchmark airport as a weighted combination of observed airports.

For airport \(i\):

$$
\lambda_1,\lambda_2,\ldots,\lambda_n \geq 0
$$

where \(\lambda_j\) determines how much airport \(j\) contributes to the benchmark.

For each input \(k\):

$$
\sum_{j=1}^{n}
\lambda_j x_{jk}
\leq
x_{ik}
$$

The benchmark therefore cannot use more input than airport \(i\).

---

## 3. Maximize proportional output expansion

Let:

$$
\phi_i \geq 1
$$

represent the maximum proportional expansion of airport \(i\)'s outputs.

For every output \(r\):

$$
\sum_{j=1}^{n}
\lambda_j y_{jr}
\geq
\phi_i y_{ir}
$$

DEA solves:

$$
\max_{\lambda,\phi} \phi_i
$$

subject to the input and output constraints.

---

# VRS DEA

## 4. Add the Variable Returns to Scale condition

For VRS:

$$
\sum_{j=1}^{n}\lambda_j = 1
$$

The complete output-oriented VRS problem is:

$$
\max_{\lambda,\phi} \phi_i
$$

subject to:

$$
\sum_{j=1}^{n}
\lambda_j x_{jk}
\leq
x_{ik}
\qquad
\forall k
$$

$$
\sum_{j=1}^{n}
\lambda_j y_{jr}
\geq
\phi_i y_{ir}
\qquad
\forall r
$$

$$
\sum_{j=1}^{n}\lambda_j = 1
$$

$$
\lambda_j \geq 0
$$

$$
\phi_i \geq 1
$$

---

## 5. Convert output expansion into VRS efficiency

The conventional DEA efficiency score is:

$$
TE_{VRS,i}
=
\frac{1}{\phi_i}
$$

Therefore:

$$
0 < TE_{VRS,i} \leq 1
$$

If:

$$
TE_{VRS,i}=1
$$

the airport is technically efficient under VRS.

If:

$$
TE_{VRS,i}<1
$$

the airport is inefficient relative to the VRS frontier.

---

# CRS DEA

## 6. Remove the VRS condition

Under Constant Returns to Scale:

$$
\sum_{j=1}^{n}\lambda_j = 1
$$

is removed.

The CRS problem is therefore:

$$
\max_{\lambda,\phi} \phi_i
$$

subject to:

$$
\sum_{j=1}^{n}
\lambda_j x_{jk}
\leq
x_{ik}
\qquad
\forall k
$$

$$
\sum_{j=1}^{n}
\lambda_j y_{jr}
\geq
\phi_i y_{ir}
\qquad
\forall r
$$

$$
\lambda_j \geq 0
$$

$$
\phi_i \geq 1
$$

The CRS efficiency score is:

$$
TE_{CRS,i}
=
\frac{1}{\phi_i}
$$

---

# Scale Efficiency

## 7. Calculate scale efficiency

Scale efficiency separates scale effects from pure technical efficiency:

$$
SE_i
=
\frac{TE_{CRS,i}}
{TE_{VRS,i}}
$$

Therefore:

$$
0 < SE_i \leq 1
$$

If:

$$
SE_i = 1
$$

the airport operates at the efficient scale.

If:

$$
SE_i < 1
$$

the airport has scale inefficiency.

---

# NIRS DEA

## 8. Estimate Non-Increasing Returns to Scale

For NIRS:

$$
\sum_{j=1}^{n}\lambda_j \leq 1
$$

The NIRS problem is:

$$
\max_{\lambda,\phi} \phi_i
$$

subject to:

$$
\sum_{j=1}^{n}
\lambda_j x_{jk}
\leq
x_{ik}
\qquad
\forall k
$$

$$
\sum_{j=1}^{n}
\lambda_j y_{jr}
\geq
\phi_i y_{ir}
\qquad
\forall r
$$

$$
\sum_{j=1}^{n}\lambda_j \leq 1
$$

$$
\lambda_j \geq 0
$$

The corresponding efficiency score is:

$$
TE_{NIRS,i}
=
\frac{1}{\phi_i}
$$

---

# Returns to Scale

## 9. Classify returns to scale

If:

$$
TE_{CRS,i}
=
TE_{VRS,i}
$$

then:

$$
RTS_i = CRS
$$

If:

$$
TE_{CRS,i}
=
TE_{NIRS,i}
$$

then:

$$
RTS_i = IRS
$$

Otherwise:

$$
RTS_i = DRS
$$

where:

- \(CRS\) = Constant Returns to Scale
- \(IRS\) = Increasing Returns to Scale
- \(DRS\) = Decreasing Returns to Scale

---

# Bootstrap Bias Correction

## 10. Generate bootstrap efficiency estimates

Repeat the DEA estimation \(B\) times to obtain:

$$
\hat{\theta}_{i}^{*(1)},
\hat{\theta}_{i}^{*(2)},
\ldots,
\hat{\theta}_{i}^{*(B)}
$$

where:

$$
\hat{\theta}_i
=
TE_{VRS,i}
$$

---

## 11. Estimate bootstrap mean efficiency

$$
\overline{\theta_i^*}
=
\frac{1}{B}
\sum_{b=1}^{B}
\hat{\theta}_i^{*(b)}
$$

---

## 12. Estimate DEA bias

$$
\widehat{Bias}_i
=
\overline{\theta_i^*}
-
\hat{\theta}_i
$$

---

## 13. Calculate bias-corrected VRS efficiency

$$
\hat{\theta}_{i,BC}
=
\hat{\theta}_i
-
\widehat{Bias}_i
$$

Equivalently:

$$
\hat{\theta}_{i,BC}
=
2\hat{\theta}_i
-
\overline{\theta_i^*}
$$

In the code:

$$
VRScor_i
=
\hat{\theta}_{i,BC}
$$

In [ ]:
airport_operational_data

,Airport,FTE,Gates,Terminal area (m²),Passengers (million),Aircraft movements (000),Cargo (000 tonnes)
0,A1,2099,66,187121,5.185255,154.301393,211.403055
1,A2,1222,51,113669,3.801233,118.023733,152.583339
2,A3,2416,80,195520,6.658260,199.677769,262.331830
3,A4,2006,58,178648,4.635913,136.229079,190.540360
4,A5,637,23,55805,1.390815,42.254945,55.632914
5,A6,2136,92,216147,4.833334,150.780379,198.350744
6,A7,1903,64,177102,5.512904,165.714933,226.153343
7,A8,1702,63,163573,4.617559,140.811980,189.435442
8,A9,588,25,47976,1.479466,46.063760,56.990700
9,A10,1372,39,114106,3.071176,90.018395,123.874299


In [ ]:
# ------------------------------------------------------------
# DEA inputs
# ------------------------------------------------------------

X = airport_operational_data.set_index("Airport")[
    [
        "FTE",
        "Gates",
        "Terminal area (m²)"
    ]
]

display(X)

,FTE,Gates,Terminal area (m²)
Airport,,,
A1,2099,66,187121
A2,1222,51,113669
A3,2416,80,195520
A4,2006,58,178648
A5,637,23,55805
A6,2136,92,216147
A7,1903,64,177102
A8,1702,63,163573
A9,588,25,47976


In [ ]:
# ------------------------------------------------------------
# DEA outputs
# ------------------------------------------------------------

Y = airport_operational_data.set_index("Airport")[
    [
        "Passengers (million)",
        "Aircraft movements (000)",
        "Cargo (000 tonnes)"
    ]
]

display(Y)

,Passengers (million),Aircraft movements (000),Cargo (000 tonnes)
Airport,,,
A1,5.185255,154.301393,211.403055
A2,3.801233,118.023733,152.583339
A3,6.658260,199.677769,262.331830
A4,4.635913,136.229079,190.540360
A5,1.390815,42.254945,55.632914
A6,4.833334,150.780379,198.350744
A7,5.512904,165.714933,226.153343
A8,4.617559,140.811980,189.435442
A9,1.479466,46.063760,56.990700


In [ ]:
airport = "A1"

airport_inputs = X.loc[airport]

airport_outputs = Y.loc[airport]

print("Airport being evaluated:", airport)

print("\nInputs:")

display(
    airport_inputs.to_frame(
        name="Value"
    )
)

print("\nOutputs:")

display(
    airport_outputs.to_frame(
        name="Value"
    )
)

Airport being evaluated: A1

Inputs:


,Value
FTE,2099
Gates,66
Terminal area (m²),187121



Outputs:


,Value
Passengers (million),5.185255
Aircraft movements (000),154.301393
Cargo (000 tonnes),211.403055


Given Airport 1's inputs, could some combination of the other airports produce more output?

##VRS

In [ ]:
# ============================================================
# Run VRS DEA for all airports
# ============================================================

# 1) creates an empty list.

vrs_scores = []

# 2) Run the DEA optimization separately for every airport wth objective: Maximize the proportional output expansion phi

for airport in X.index:

    n = len(X)
    c = np.r_[
        np.zeros(n),
        -1
    ]


# 3) The synthetic benchmark cannot use more inputs than the airport being evaluated​
    A_ub = []
    b_ub = []

    for column in X.columns:

        A_ub.append(
            np.r_[
                X[column].to_numpy(dtype=float),
                0
            ]
        )

        b_ub.append(
            X.loc[airport, column]
        )


# 4) the synthetic benchmark must produce at least phi times the outputs of the airport being evaluated
    for column in Y.columns:

        A_ub.append(
            np.r_[
                -Y[column].to_numpy(dtype=float),
                Y.loc[airport, column]
            ]
        )

        b_ub.append(0)


# 5) VRS condition Sum of lambda weights = 1

    A_eq = np.array([
        np.r_[
            np.ones(n),
            0
        ]
    ])

    b_eq = np.array([1])

# 6) Solve linear program

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        A_eq=A_eq,
        b_eq=b_eq,
        bounds=[(0, None)] * n + [(1, None)],
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            f"DEA failed for {airport}: {result.message}"
        )

# 7) Convert phi to conventional efficiency score

    phi = result.x[-1]

    vrs_efficiency = 1 / phi

    vrs_scores.append(
        vrs_efficiency
    )



In [ ]:
# ============================================================
# Add VRS results to airport_operational_data
# ============================================================

airport_operational_data["VRS"] = vrs_scores


# ============================================================
# Display full VRS results
# ============================================================

display(
    airport_operational_data[
        [
            "Airport",
            "VRS"
        ]
    ]
    .style
    .format({
        "VRS": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption("VRS DEA Efficiency")
)

Airport,VRS
A1,0.8538
A2,1.0000
A3,1.0000
A4,0.7947
A5,0.9076
A6,0.8223
A7,0.9760
A8,0.8959
A9,1.0000
A10,0.8082


##CRS

VRS asks how well the airport operates at its scale. CRS additionally asks whether it is operating at the right scale.

In [ ]:
# ============================================================
# Run CRS DEA for all airports
# ============================================================

crs_scores = []

for airport in X.index:

    n = len(X)

    c = np.r_[
        np.zeros(n),
        -1
    ]

    A_ub = []
    b_ub = []

    for column in X.columns:

        A_ub.append(
            np.r_[
                X[column].to_numpy(dtype=float),
                0
            ]
        )

        b_ub.append(
            X.loc[airport, column]
        )

    for column in Y.columns:

        A_ub.append(
            np.r_[
                -Y[column].to_numpy(dtype=float),
                Y.loc[airport, column]
            ]
        )

        b_ub.append(0)


    # --------------------------------------------------------
    # CRS:
    #
    # No sum(lambda) = 1 constraint
    # --------------------------------------------------------

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        bounds=[(0, None)] * n + [(1, None)],
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            f"DEA failed for {airport}: {result.message}"
        )

    phi = result.x[-1]

    crs_scores.append(
        1 / phi
    )


In [ ]:

# ============================================================
# Add CRS results
# ============================================================

airport_operational_data["CRS"] = crs_scores

In [ ]:
# ============================================================
# Calculate scale efficiency
# ============================================================

airport_operational_data["Scale efficiency"] = (
    airport_operational_data["CRS"]
    / airport_operational_data["VRS"]
)

In [ ]:

# ============================================================
# Display results
# ============================================================

display(
    airport_operational_data[
        [
            "Airport",
            "VRS",
            "CRS",
            "Scale efficiency"
        ]
    ]
    .style
    .format({
        "VRS": "{:.4f}",
        "CRS": "{:.4f}",
        "Scale efficiency": "{:.4f}"
    })
    .hide(axis="index")
)

Airport,VRS,CRS,Scale efficiency
A1,0.8538,0.8109,0.9498
A2,1.0000,0.9482,0.9482
A3,1.0000,0.9640,0.9640
A4,0.7947,0.7723,0.9719
A5,0.9076,0.7043,0.7760
A6,0.8223,0.7046,0.8569
A7,0.9760,0.9074,0.9297
A8,0.8959,0.8436,0.9416
A9,1.0000,0.8432,0.8432
A10,0.8082,0.7928,0.9809


##NIRS

In [ ]:
# ============================================================
# Run NIRS DEA for all airports
# ============================================================

nirs_scores = []

for airport in X.index:

    n = len(X)

    # --------------------------------------------------------
    # Objective:
    # maximize proportional output expansion phi
    #
    # scipy minimizes, so minimize -phi
    # --------------------------------------------------------

    c = np.r_[
        np.zeros(n),
        -1
    ]

    A_ub = []
    b_ub = []


    # --------------------------------------------------------
    # Input constraints
    # --------------------------------------------------------

    for column in X.columns:

        A_ub.append(
            np.r_[
                X[column].to_numpy(dtype=float),
                0
            ]
        )

        b_ub.append(
            X.loc[airport, column]
        )


    # --------------------------------------------------------
    # Output constraints
    # --------------------------------------------------------

    for column in Y.columns:

        A_ub.append(
            np.r_[
                -Y[column].to_numpy(dtype=float),
                Y.loc[airport, column]
            ]
        )

        b_ub.append(0)


    # --------------------------------------------------------
    # NIRS condition:
    #
    # sum(lambda) <= 1
    # --------------------------------------------------------

    A_ub.append(
        np.r_[
            np.ones(n),
            0
        ]
    )

    b_ub.append(1)


    # --------------------------------------------------------
    # Solve linear program
    # --------------------------------------------------------

    result = linprog(
        c,
        A_ub=np.array(A_ub),
        b_ub=np.array(b_ub),
        bounds=[(0, None)] * n + [(1, None)],
        method="highs"
    )

    if not result.success:
        raise RuntimeError(
            f"NIRS DEA failed for {airport}: {result.message}"
        )


    # --------------------------------------------------------
    # Convert phi to conventional efficiency score
    # --------------------------------------------------------

    phi = result.x[-1]

    nirs_scores.append(
        1 / phi
    )


# ============================================================
# Add NIRS results
#
# We need NIRS only to determine whether scale inefficiency
# reflects increasing or decreasing returns to scale.
# ============================================================

airport_operational_data["NIRS"] = nirs_scores


# ============================================================
# Scale efficiency
#
# CRS/VRS tells us how much scale inefficiency exists.
# ============================================================

airport_operational_data["CRS/VRS"] = (
    airport_operational_data["CRS"]
    / airport_operational_data["VRS"]
)


# ============================================================
# Returns to scale classification
#
# C = Constant returns to scale
# I = Increasing returns to scale
# D = Decreasing returns to scale
# ============================================================

def classify_rts(row):

    # CRS = VRS
    # -> no scale inefficiency
    if np.isclose(
        row["CRS"],
        row["VRS"],
        atol=1e-6
    ):
        return "C"

    # CRS = NIRS
    # -> increasing returns to scale
    elif np.isclose(
        row["CRS"],
        row["NIRS"],
        atol=1e-6
    ):
        return "I"

    # Otherwise NIRS = VRS
    # -> decreasing returns to scale
    else:
        return "D"


airport_operational_data["CRS/NIRS"] = airport_operational_data.apply(
    classify_rts,
    axis=1
)



In [ ]:
# ============================================================
# Display results
# ============================================================

display(
    airport_operational_data[
        [
            "Airport",
            "VRS",
            "CRS/VRS",
            "CRS/NIRS"
        ]
    ]
    .style
    .format({
        "VRS": "{:.4f}",
        "CRS/VRS": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption(
        "DEA Returns to Scale Results"
    )
)

Airport,VRS,CRS/VRS,CRS/NIRS
A1,0.8538,0.9498,D
A2,1.0000,0.9482,D
A3,1.0000,0.9640,D
A4,0.7947,0.9719,D
A5,0.9076,0.7760,I
A6,0.8223,0.8569,D
A7,0.9760,0.9297,D
A8,0.8959,0.9416,D
A9,1.0000,0.8432,I
A10,0.8082,0.9809,I


##VRScore

In [ ]:
B = 1000

rng_boot = np.random.default_rng(123)

X_np = X.to_numpy(dtype=float)
Y_np = Y.to_numpy(dtype=float)

vrs_original = airport_operational_data["VRS"].to_numpy(dtype=float)

n = len(airport_operational_data)


# ============================================================
# Function:
# Calculate VRS efficiency relative to a reference sample
# ============================================================

def calculate_vrs_bootstrap(
    X_reference,
    Y_reference,
    X_evaluate,
    Y_evaluate
):

    scores = []

    n_reference = len(X_reference)

    for i in range(len(X_evaluate)):

        # ----------------------------------------------------
        # Objective:
        # maximize phi
        # scipy minimizes, therefore minimize -phi
        # ----------------------------------------------------

        c = np.r_[
            np.zeros(n_reference),
            -1
        ]


        A_ub = []
        b_ub = []


        # ----------------------------------------------------
        # Input constraints
        # ----------------------------------------------------

        for j in range(X_reference.shape[1]):

            A_ub.append(
                np.r_[
                    X_reference[:, j],
                    0
                ]
            )

            b_ub.append(
                X_evaluate[i, j]
            )


        # ----------------------------------------------------
        # Output constraints
        # ----------------------------------------------------

        for r in range(Y_reference.shape[1]):

            A_ub.append(
                np.r_[
                    -Y_reference[:, r],
                    Y_evaluate[i, r]
                ]
            )

            b_ub.append(0)


        # ----------------------------------------------------
        # VRS condition
        #
        # sum(lambda) = 1
        # ----------------------------------------------------

        A_eq = np.array([
            np.r_[
                np.ones(n_reference),
                0
            ]
        ])

        b_eq = np.array([1])


        # ----------------------------------------------------
        # Solve
        #
        # phi is allowed below 1 in bootstrap samples
        # ----------------------------------------------------

        result = linprog(
            c,
            A_ub=np.array(A_ub),
            b_ub=np.array(b_ub),
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=
                [(0, None)] * n_reference
                + [(1e-10, None)],
            method="highs"
        )

        if not result.success:
            raise RuntimeError(
                result.message
            )


        phi = result.x[-1]

        scores.append(
            1 / phi
        )


    return np.array(scores)

In [ ]:
# ============================================================
# Smooth bootstrap distribution of VRS efficiencies
# ============================================================

mean_vrs = np.mean(
    vrs_original
)

sd_vrs = np.std(
    vrs_original,
    ddof=1
)

variance_vrs = np.var(
    vrs_original,
    ddof=1
)


# ------------------------------------------------------------
# Smoothing bandwidth
# ------------------------------------------------------------

h = (
    1.06
    * sd_vrs
    * n ** (-1 / 5)
)


# ------------------------------------------------------------
# Store bootstrap DEA scores
# ------------------------------------------------------------

bootstrap_scores = np.zeros(
    (B, n)
)

In [ ]:
# ============================================================
# Run homogeneous smooth bootstrap
# ============================================================

for b in range(B):

    sampled_vrs = rng_boot.choice(
        vrs_original,
        size=n,
        replace=True
    )

    smoothed_vrs = (
        sampled_vrs
        + h
        * rng_boot.normal(
            size=n
        )
    )

    smoothed_vrs = np.where(
        smoothed_vrs > 1,
        2 - smoothed_vrs,
        smoothed_vrs
    )

    bootstrap_efficiency = (
        mean_vrs
        +
        (
            smoothed_vrs
            - mean_vrs
        )
        /
        np.sqrt(
            1
            + h**2
            / variance_vrs
        )
    )


    bootstrap_efficiency = np.maximum(
        bootstrap_efficiency,
        1e-10
    )

    Y_star = (
        bootstrap_efficiency
        / vrs_original
    )[:, None] * Y_np

    bootstrap_scores[b, :] = (
        calculate_vrs_bootstrap(
            X_np,
            Y_star,
            X_np,
            Y_np
        )
    )

In [ ]:
# ============================================================
# Bias correction
# ============================================================

mean_bootstrap_vrs = (
    bootstrap_scores.mean(
        axis=0
    )
)

vrs_bias = (
    mean_bootstrap_vrs
    - vrs_original
)

vrs_corrected = (
    vrs_original
    - vrs_bias
)


airport_operational_data["VRScor"] = (
    vrs_corrected
)

In [ ]:
# ============================================================
# Display original and corrected VRS
# ============================================================

display(
    airport_operational_data[
        [
            "Airport",
            "VRS",
            "VRScor"
        ]
    ]
    .style
    .format({
        "VRS": "{:.4f}",
        "VRScor": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption(
        "Original and Bias-Corrected VRS Efficiency"
    )
)

Airport,VRS,VRScor
A1,0.8538,0.8078
A2,1.0000,0.9476
A3,1.0000,0.9062
A4,0.7947,0.7496
A5,0.9076,0.8526
A6,0.8223,0.7854
A7,0.9760,0.9351
A8,0.8959,0.8581
A9,1.0000,0.8812
A10,0.8082,0.7627


In [ ]:
# ============================================================
# Compare original and bias-corrected DEA scores
# ============================================================

dea_results = (
    airport_operational_data[
        [
            "Airport",
            "VRS",
            "VRScor"
        ]
    ]
    .sort_values(
        "VRScor",
        ascending=False
    )
    .reset_index(drop=True)
)

dea_results["Rank"] = (
    dea_results.index + 1
)

display(
    dea_results[
        [
            "Rank",
            "Airport",
            "VRS",
            "VRScor"
        ]
    ]
    .style
    .format({
        "VRS": "{:.4f}",
        "VRScor": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption(
        "Original and Bias-Corrected VRS Efficiency"
    )
)

Rank,Airport,VRS,VRScor
1,A13,1.0000,0.9503
2,A2,1.0000,0.9476
3,A12,0.9874,0.9417
4,A7,0.9760,0.9351
5,A11,0.9642,0.9129
6,A3,1.0000,0.9062
7,A17,0.9387,0.8952
8,A20,1.0000,0.8904
9,A16,1.0000,0.8867
10,A18,1.0000,0.8861


##Summary

In [ ]:
dea_final = airport_operational_data[
    [
        "Airport",
        "VRS",
        "VRScor",
        "CRS/VRS",
        "CRS/NIRS"
    ]
].copy()


display(
    dea_final
    .style
    .format({
        "VRS": "{:.4f}",
        "VRScor": "{:.4f}",
        "CRS/VRS": "{:.4f}"
    })
    .hide(axis="index")
    .set_caption(
        "Final DEA Results"
    )
)

Airport,VRS,VRScor,CRS/VRS,CRS/NIRS
A1,0.8538,0.8078,0.9498,D
A2,1.0000,0.9476,0.9482,D
A3,1.0000,0.9062,0.9640,D
A4,0.7947,0.7496,0.9719,D
A5,0.9076,0.8526,0.7760,I
A6,0.8223,0.7854,0.8569,D
A7,0.9760,0.9351,0.9297,D
A8,0.8959,0.8581,0.9416,D
A9,1.0000,0.8812,0.8432,I
A10,0.8082,0.7627,0.9809,I
